In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### anti mammalian cells dataset integration and label-consistency analysis
- This notebook integrates peptide-level anti mammalian cells annotations derived from PepNet and iAMPCN into a unified, curated dataset. The input consists of a preprocessed CSV file containing peptide sequences and binary anti mammalian cells labels.

- All unique peptide sequences are collected and used to construct a pivot table in which each row represents a unique sequence and each column corresponds to a data source. Since only one source is available, the pivot structure is kept consistent with other toxicity tasks to ensure methodological uniformity.

- Sequence-level quality control is applied by removing peptides containing non-canonical amino acids and by filtering sequences outside predefined minimum and maximum length thresholds. Statistics describing the impact of these filters and the resulting length distribution are recorded.

- Source-specific labels are mapped onto the pivot table using a standardized encoding scheme that distinguishes positive, negative, unlabeled, and unknown annotations. Label consistency is evaluated by counting per-sequence label occurrences and computing the proportion of positive and negative evidence.

- Based on this analysis, sequences are categorized into exclusive positive, exclusive negative, unlabeled-only, or ambiguous groups. Ambiguous cases are further stratified according to the percentage of positive annotations to provide a graded confidence assessment.

- Finally, curated output subsets and comprehensive metadata are generated and exported in a structured format, enabling reproducible downstream analysis and direct use in machine learning workflows.

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/anti_mammalian_cells"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_iAMPCN_anti_mammalian_cells = pd.read_csv(f"{PATH_EXPORT}/{name_task}/iAMPCN/processed_anti_mammalian_cells_dataset.csv")
df_iAMPCN_anti_mammalian_cells = df_iAMPCN_anti_mammalian_cells.rename(columns={"label": "anti_mammalian_cells"})

In [4]:
df_PepNet_anti_mammalian_cells = pd.read_csv(f"{PATH_EXPORT}/{name_task}/PepNet/processed_anti_mammalian_cells_dataset.csv")
df_PepNet_anti_mammalian_cells = df_PepNet_anti_mammalian_cells.rename(columns={"label": "anti_mammalian_cells"})

- Collecting all sequences for activity

In [5]:
df_list_anti_mammalian_cells = [
    df_iAMPCN_anti_mammalian_cells,
    df_PepNet_anti_mammalian_cells
]
unique_sequence_anti_mammalian_cells = count_unique_sequence(df_list_anti_mammalian_cells)

22789


- Create pivote dataset

In [6]:
df_pivote = create_pivote(unique_sequence_anti_mammalian_cells)

- Removing sequences with non canonical residues 

In [7]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [8]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True     22787
False        2
Name: count, dtype: int64


- Filter sequences by length

In [9]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    22787.000000
mean        28.860008
std         31.300671
min          5.000000
25%         14.000000
50%         20.000000
75%         33.000000
max        692.000000
Name: length, dtype: float64

In [10]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [11]:
df_pivote["filter_length"].value_counts()

filter_length
True     21393
False     1394
Name: count, dtype: int64

In [12]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [13]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(21393, 4)

In [14]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [15]:
df_list_anti_mammalian_cells = [("iAMPCN", df_iAMPCN_anti_mammalian_cells),
                    ("PepNet", df_PepNet_anti_mammalian_cells)]

In [16]:
for source, dataset in df_list_anti_mammalian_cells:
    dataset = dataset[["sequence", "anti_mammalian_cells"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["anti_mammalian_cells"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [17]:
df_pivote

,sequence,iAMPCN,PepNet
0,RTIQVENSHLILTGAGA,0,0
1,FLGWLFKWAKK,0,0
2,GVLGTVKNLLIGAGKSAAQSVLKTLSCKLSNDC,1,1
3,NQGRHFSGGALIHARFVMTAASCFQ,0,0
4,KTNDRAAIGKLVGATSSL,0,0
...,...,...,...
22784,RRWRITVIRVRR,0,0
22785,ILPFVAGVAAEMMQHVYCAASKKC,0,0
22786,MPADILETRTSETEDLLDLDLSIGVEEITAGPAVTSWSLCTPGCTS...,0,0
22787,VLTMGKPLTAMPGLTRDGVLGKWIWQLAE,0,0


- Working with pivote for detecting ambiguous sequences 

In [18]:
df_pivote = process_count_labels(df_pivote)

In [19]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
True     17729
False     3664
Name: count, dtype: int64

In [20]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
True     17729
False     3664
Name: count, dtype: int64

In [21]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
False    17729
True      3664
Name: count, dtype: int64

In [22]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
False    17729
True      3664
Name: count, dtype: int64

In [23]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    21393
Name: count, dtype: int64

In [24]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,iAMPCN,PepNet,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
2,GVLGTVKNLLIGAGKSAAQSVLKTLSCKLSNDC,1,1,2,0,0,0,True,False,True,False,False,0.0,100.0
6,FLGMIPGLIGGLISAFK,1,1,2,0,0,0,True,False,True,False,False,0.0,100.0
9,FKLRAKIKVRLRAKIKL,1,1,2,0,0,0,True,False,True,False,False,0.0,100.0
22744,RRRFFF,1,1,2,0,0,0,True,False,True,False,False,0.0,100.0
22747,FLPIAGKLLSGLSGLL,1,1,2,0,0,0,True,False,True,False,False,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22783,FVGSLASKVVPSVFGAIKTK,0,0,0,2,0,0,False,True,False,True,False,100.0,0.0
22784,RRWRITVIRVRR,0,0,0,2,0,0,False,True,False,True,False,100.0,0.0
22785,ILPFVAGVAAEMMQHVYCAASKKC,0,0,0,2,0,0,False,True,False,True,False,100.0,0.0
22786,MPADILETRTSETEDLLDLDLSIGVEEITAGPAVTSWSLCTPGCTS...,0,0,0,2,0,0,False,True,False,True,False,100.0,0.0


- Splitting data into only negative, only positive, and with amiguous data

In [25]:
negative = df_pivote[df_pivote["negative"]]

In [26]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [27]:
positive = df_pivote[df_pivote["positive"]]

In [28]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [29]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [30]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [31]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [32]:
df_ambiguous["Category_pbb"].value_counts()

Series([], Name: count, dtype: int64)

- Working with metada

In [33]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="anti_mammalian_cells",
    source_list=df_list_anti_mammalian_cells,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)
metadata

{'task': 'anti_mammalian_cells',
 'generated_at': '2026-09-04T20:35:58.365849',
 'sources': {'n_unique_sequences': {'iAMPCN': 22789, 'PepNet': 22787}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 22789, 'after': 22787},
  'length_filter': {'before': 22787, 'after': 21393},
  'length_distribution': {'min': 5, 'max': 70, 'mean': 23.52, 'median': 20.0}},
 'statistics': {'total_sequences_final': 21393,
  'positive': {'positive_and_unlabel': 3664, 'only_positive': 3664},
  'negative': {'negative_and_unlabel': 17729, 'only_negative': 17729},
  'only_unlabel': 0,
  'ambiguous': {'n_sequences': 0}}}

- Exporting data

In [34]:
os.makedirs(output_folder, exist_ok=True)

In [35]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [36]:
negative.shape

(17729, 14)

In [37]:
only_negative.shape

(17729, 14)

In [38]:
positive.shape

(3664, 14)

In [39]:
only_positive.shape

(3664, 14)

In [40]:
only_unlabel.shape

(0, 14)

In [41]:
df_ambiguous.shape

(0, 15)

In [42]:
negative.to_csv(f"{output_folder}/negative.csv", index=False)

In [43]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)